In [5]:
%pip install torch transformers datasets pandas numpy
%pip install datasets transformers torch

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [6]:
from datasets import load_dataset
from transformers import AutoTokenizer
import torch

In [7]:
dataset = load_dataset("rajpurkar/squad")


Generating validation split: 100%|██████████| 10570/10570 [00:00<00:00, 1072340.98 examples/s]


In [8]:
# Check the structure of the dataset
print(dataset)

# Access the train and validation sets
train_dataset = dataset['train']
validation_dataset = dataset['validation']

# Inspect a sample from the train set
print(train_dataset[0])


DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})
{'id': '5733be284776f41900661182', 'title': 'University_of_Notre_Dame', 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome

In [19]:
# import pandas as pd

# # Convert train dataset to DataFrame
# train_df = pd.DataFrame(dataset['train'])

# # Sample a subset of data (e.g., 5k rows)
# train_df = train_df.sample(n=5000, random_state=42)

# # Extract question and answer columns in a cleaner format
# questions = train_df['question'].tolist()
# answers = [ans['text'][0] for ans in train_df['answers']]  # Extract answer text from the 'answers' field

# # Create a new DataFrame with question-answer pairs
# qa_df = pd.DataFrame({
#     'question': questions,
#     'answer': answers
# })

# # Display the first few rows to see the cleaned format
# print(qa_df.head())

# Extract questions and answers in a cleaner format
questions = [entry['question'] for entry in train_dataset]
answers = [entry['answers']['text'][0] for entry in train_dataset]  # Using the first answer if there are multiple answers

# Create a new DataFrame with question-answer pairs
import pandas as pd
qa_df = pd.DataFrame({
    'question': questions,
    'answer': answers
})

# Display the first few rows to see the cleaned format
print(qa_df.head())




                                            question  \
0  To whom did the Virgin Mary allegedly appear i...   
1  What is in front of the Notre Dame Main Building?   
2  The Basilica of the Sacred heart at Notre Dame...   
3                  What is the Grotto at Notre Dame?   
4  What sits on top of the Main Building at Notre...   

                                    answer  
0               Saint Bernadette Soubirous  
1                a copper statue of Christ  
2                        the Main Building  
3  a Marian place of prayer and reflection  
4       a golden statue of the Virgin Mary  


In [21]:
# Load the pre-trained BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Tokenize the questions and answers
qa_df['question_tokens'] = qa_df['question'].apply(lambda x: tokenizer.encode(x, truncation=True, padding='max_length', max_length=32))
qa_df['answer_tokens'] = qa_df['answer'].apply(lambda x: tokenizer.encode(x, truncation=True, padding='max_length', max_length=32))

# Check the first few tokenized questions and answers
print(qa_df[['question_tokens', 'answer_tokens']].head())


                                     question_tokens  \
0  [101, 2000, 3183, 2106, 1996, 6261, 2984, 9382...   
1  [101, 2054, 2003, 1999, 2392, 1997, 1996, 1028...   
2  [101, 1996, 13546, 1997, 1996, 6730, 2540, 201...   
3  [101, 2054, 2003, 1996, 24665, 23052, 2012, 10...   
4  [101, 2054, 7719, 2006, 2327, 1997, 1996, 2364...   

                                       answer_tokens  
0  [101, 3002, 16595, 9648, 4674, 2061, 12083, 97...  
1  [101, 1037, 6967, 6231, 1997, 4828, 102, 0, 0,...  
2  [101, 1996, 2364, 2311, 102, 0, 0, 0, 0, 0, 0,...  
3  [101, 1037, 14042, 2173, 1997, 7083, 1998, 918...  
4  [101, 1037, 3585, 6231, 1997, 1996, 6261, 2984...  


In [22]:
# Convert tokenized questions and answers into PyTorch tensors
questions_tensor = torch.tensor(qa_df['question_tokens'].tolist())
answers_tensor = torch.tensor(qa_df['answer_tokens'].tolist())

In [27]:
from torch.utils.data import DataLoader, TensorDataset

# Create a TensorDataset
dataset = TensorDataset(questions_tensor, answers_tensor)

# Create a DataLoader for batching
batch_size = 8
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Check the first batch to ensure everything is correct
for batch in data_loader:
    print(batch)
    break


[tensor([[  101,  2054,  3820,  2170,  2005,  1996,  2644,  1997, 17601,  1029,
           102,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0],
        [  101,  2054,  2001,  1996,  2220,  2171,  2005,  1996,  9433,  4429,
          1029,   102,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0],
        [  101,  2043,  2515,  2087, 26043,  5258,  1029,   102,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0],
        [  101,  2040,  2001,  5827,  2007,  2759,  6026,  1996,  2224,  1997,
          2171, 14403, 17151, 21493,  2483,  2000,  2660,  1029,   102,     0,
             0,     0,     0,     0,     0,     0,     0, 

In [28]:
import torch.nn as nn
import torch.optim as optim

class QuestionAnsweringModel(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super(QuestionAnsweringModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.RNN(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = self.fc(out)
        return out

# Hyperparameters
vocab_size = len(tokenizer.vocab)  # Size of the tokenizer's vocabulary
embed_size = 128                   # Size of the embedding layer
hidden_size = 64                   # Size of the hidden layer

# Create the model instance
model = QuestionAnsweringModel(vocab_size, embed_size, hidden_size)

# Loss function and optimizer
loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)  # Ignore padding tokens
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [29]:
# Training loop
def train_model(model, data_loader, optimizer, loss_fn, epochs=5):
    model.train()  # Set model to training mode
    for epoch in range(epochs):
        total_loss = 0
        for questions_batch, answers_batch in data_loader:
            optimizer.zero_grad()

            # Forward pass
            output = model(questions_batch)
            output = output.view(-1, vocab_size)  # Flatten the output to (batch_size * seq_len, vocab_size)
            answers_batch = answers_batch.view(-1)  # Flatten answers to match the output shape

            # Calculate the loss
            loss = loss_fn(output, answers_batch)
            loss.backward()  # Backpropagation
            optimizer.step()  # Update the weights

            total_loss += loss.item()

        print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss / len(data_loader)}")

# Train the model
train_model(model, data_loader, optimizer, loss_fn)


Epoch [1/5], Loss: 5.770735455473808


KeyboardInterrupt: 

In [ ]:
def evaluate_model(model, data_loader):
    model.eval()  # Set the model to evaluation mode
    correct = 0
    total = 0

    with torch.no_grad():  # No need to track gradients during evaluation
        for questions_batch, answers_batch in data_loader:
            # Forward pass
            output = model(questions_batch)
            output = output.argmax(dim=2)  # Get the predicted tokens with the highest probability
            output = output.view(-1).cpu().numpy()  # Flatten and move to CPU for comparison

            answers_batch = answers_batch.view(-1).cpu().numpy()  # Flatten answers for comparison

            correct += (output == answers_batch).sum()
            total += len(answers_batch)

    accuracy = correct / total
    print(f"Accuracy: {accuracy * 100:.2f}%")

# Evaluate the model
evaluate_model(model, data_loader)
